# Ablation 2: Rebalancing Only (NO SSL) — Kaggle version, macro-F1 checkpoint selection

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import shutil
import random
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import json

# =====================================================================
# 1. SETUP & PATH CONFIGURATION (KAGGLE PATHS)
# =====================================================================
# Converted from the original Colab version, which mounted Google Drive.
# Same Kaggle input/output layout as the main SSL+rebalance and
# ResNet-18 baseline notebooks, so all outputs are directly comparable.

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
UNLABELED_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'

OUTPUT_DIR = '/kaggle/working/Thesis_Outputs'
LABELED_DIR = '/kaggle/working/data/labeled_real'

os.makedirs(LABELED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(UNLABELED_DIR):
    unlabeled_count = len([f for f in os.listdir(UNLABELED_DIR) if f.endswith('.jpg')])
    print(f"Verified dataset. Found {unlabeled_count} images in Kaggle source input directory.")
else:
    raise FileNotFoundError(f"Could not locate image directory at {UNLABELED_DIR}")


Device: cuda
Verified dataset. Found 25331 images in Kaggle source input directory.


**SETUP & EXTRACTION** — filter labeled images into the writable working directory

In [2]:
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

found = 0
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        found += 1

print(f"Copied/verified {found}/{len(all_images)} labeled images into working directory.")


Copied/verified 691/691 labeled images into working directory.


**RANDOMLY INITIALIZED ENCODER (NO SSL pre-training)**

In [3]:
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

## RANDOM initialization — NO pre-training
encoder = SimpleEncoder()
print("Randomly initialized encoder (NO SSL pre-training)")

## Unfreezing all
for param in encoder.parameters():
    param.requires_grad = True

model = SSLClassifier(encoder, 3).to(device)


Randomly initialized encoder (NO SSL pre-training)


**AUGMENTED DATASET (MEL oversampling — same as main experiment)**

In [4]:
normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, mel_multiplier=5, transform_normal=None, transform_mel=None):
        self.df = df
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True)\
                          .sample(frac=1, random_state=SEED).reset_index(drop=True)

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img = Image.open(f"{LABELED_DIR}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
        return img, label

train_ds = AugmentedDataset(train_df, 5, normal_transform, mel_transform)
val_ds = AugmentedDataset(val_df, 1, test_transform, test_transform)
test_ds = AugmentedDataset(test_df, 1, test_transform, test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Train: {len(train_ds)} (with MEL 5x oversampling)")
print(f"Class distribution: {train_df['label'].value_counts().to_dict()}")


Train: 623 (with MEL 5x oversampling)
Class distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}


## **FOCAL LOSS + CLASS WEIGHTS (rebalancing enabled)**

In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
weights_list = [weight_dict[cls] for cls in train_ds.classes]
weights = torch.tensor(weights_list, dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=weights, gamma=1.5)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print(f"\nFocal Loss with weights: {dict(zip(train_ds.classes, weights_list))}")



Focal Loss with weights: {'BKL': 2.0, 'MEL': 4.0, 'NV': 1.0}


# **Training & Evaluation**

**CHECKPOINT-SELECTION FIX (per supervisor review):** the original ablation kept
whichever epoch had the highest plain validation *accuracy*. That criterion is
inconsistent with a focal-loss-trained, imbalance-aware pipeline and can pick an
epoch with inflated MEL recall almost by chance. This version selects the
checkpoint by validation **macro-F1** instead, matching the corrected criterion
used in the main SSL+rebalance and ResNet-18 baseline notebooks. Validation
accuracy is still logged every epoch for comparison, it is just no longer the
selection criterion. Kept single-seed (seed 42) to match the original ablation
budget.

In [6]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, macro_f1, all_labels, all_preds

print(f"\n{'='*37}")
print("ABLATION 2: Rebalancing Only (NO SSL)")
print(f"{'='*37}")

epochs = 30
best_val_macro_f1 = -1.0
patience = 7
epochs_no_improve = 0
best_epoch = -1
ckpt_path = f'{OUTPUT_DIR}/best_rebalance_only.pth'
history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_macro_f1, _, _ = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), ckpt_path)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val Acc={val_acc:.3f}, Val Macro-F1={val_macro_f1:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest checkpoint: epoch {best_epoch} (Val Macro-F1={best_val_macro_f1:.3f})")

### Evaluate
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_acc, test_macro_f1, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*37)
print("Results: Rebalancing Only (NO SSL)")
print("="*37)
print(f"Test Accuracy: {test_acc:.3f}")
print(f"Test Macro-F1: {test_macro_f1:.3f}")

report = classification_report(true_labels, pred_labels,
                                target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n===>>> MEL Recall: {mel_recall:.1%} <<<===")
print(f"Predictions: {Counter(pred_labels)}")

## Save
results = {
    'experiment': 'Rebalance_Only_No_SSL',
    'seed': SEED,
    'ssl': False,
    'rebalancing': True,
    'augmentation': True,
    'checkpoint_selection_metric': 'val_macro_f1',
    'best_epoch': best_epoch,
    'best_val_macro_f1': best_val_macro_f1,
    'test_accuracy': test_acc,
    'test_macro_f1': test_macro_f1,
    'mel_recall': mel_recall,
    'report': report,
    'history': history,
}
with open(f'{OUTPUT_DIR}/rebalance_only_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f'{OUTPUT_DIR}/rebalance_only_finetuned.pth')
print(f"\nSaved results and weights to {OUTPUT_DIR}")



ABLATION 2: Rebalancing Only (NO SSL)
Epoch 3: Train=0.544, Val Acc=0.548, Val Macro-F1=0.381
Epoch 6: Train=0.629, Val Acc=0.587, Val Macro-F1=0.410
Epoch 9: Train=0.650, Val Acc=0.635, Val Macro-F1=0.446
Epoch 12: Train=0.665, Val Acc=0.577, Val Macro-F1=0.450
Epoch 15: Train=0.695, Val Acc=0.615, Val Macro-F1=0.454
Epoch 18: Train=0.693, Val Acc=0.644, Val Macro-F1=0.471
Epoch 21: Train=0.674, Val Acc=0.635, Val Macro-F1=0.458
Epoch 24: Train=0.687, Val Acc=0.606, Val Macro-F1=0.418
Early stopping at epoch 24

Best checkpoint: epoch 17 (Val Macro-F1=0.477)

Results: Rebalancing Only (NO SSL)
Test Accuracy: 0.673
Test Macro-F1: 0.536
              precision    recall  f1-score   support

         BKL       0.43      0.62      0.51        21
         MEL       0.27      0.38      0.32         8
          NV       0.86      0.72      0.78        75

    accuracy                           0.67       104
   macro avg       0.52      0.57      0.54       104
weighted avg       0.73      